In [1]:
def readctm(filename):
    lines = []
    with open(filename) as inf:
        for line in inf.readlines():
            lines.append(line.strip().split(" "))
    return lines

In [37]:
def strctm(text):
    return [line.strip().split(" ") for line in text.split("\n") if line != ""]

In [2]:
def normword(word):
    if word.endswith("..."):
        word = word[0:-3]
    if word[-1:] in ".,?!-:;":
        word = word[0:-1]
    return word.lower()

In [32]:
def push_back_ins(ctm_lines, start, end, offset):
    pre = []
    post = []
    buf = []

    for line in ctm_lines:
        if float(line[2]) < start:
            pre.append(line)
        elif float(line[2]) <= end:
            buf.append(line)
        else:
            post.append(line)

    i = 0
    while i < (len(buf) - offset):
        print(buf[i], buf[i+offset])
        replace = buf[i+offset][6]
        replacenorm = normword(replace)
        buf[i][4] = replacenorm
        buf[i][6] = replace
        buf[i][7] = "cor"
        i += 1
    while i < len(buf):
        buf[i][6] = "-"
        buf[i][7] = "-"
        i += 1
        
    return pre + buf + post

In [56]:
def merge_left(ctmlines, replacement = None):
    outword = ctmlines[-1][6]
    if replacement is not None:
        outword = replacement
    out = ctmlines[0][0:3]
    out += [ctmlines[-1][3], normword(outword), "1.0", outword, "cor"]
    return out

In [33]:
ctmlines = readctm("/tmp/ctmedit/hsi_1_0515_209_001_inter.txt")

In [60]:
def printctm(toprint):
    for ll in toprint:
        print(" ".join(ll))

In [67]:
def as_csv(c):
    print(f"{c[2]}\t{c[3]}\t{c[6]}")

In [106]:
from pathlib import Path

CTMDIR = Path("/Users/joregan/Playing/hsi_ctmedit")

for ctmfile in CTMDIR.glob("*.txt"):
    ctmlines = readctm(str(ctmfile))
    outlines = []
    for ctmline in ctmlines:
        if len(ctmline) < 7:
            outlines.append(ctmline)
            continue
        if ctmline[7] != "sub":
            outlines.append(ctmline)
        else:
            if ctmline[4] == normword(ctmline[6]):
                ctmline[7] = "cor"
            outlines.append(ctmline)
    with open(str(ctmfile), "w") as outf:
        for line in outlines:
            outf.write(" ".join(line) + "\n")

80008 cor
22110 sub
11758 ins
7578 del

In [75]:
from pathlib import Path

CTMDIR = Path("/Users/joregan/Playing/hsi_ctmedit")

for ctmfile in CTMDIR.glob("*.txt"):
    ctmlines = readctm(str(ctmfile))
    outlines = []
    for ctmline in ctmlines:
        if len(ctmline) < 7:
            outlines.append(ctmline)
            continue
        if ctmline[7] != "ins":
            outlines.append(ctmline)
        else:
            if ctmline[4] in ["ah", "eh", "oh", "mhm", "m", "um"]:
                ctmline[7] = "cor"
                ctmline[6] = f"[{ctmline[4]}]"
            outlines.append(ctmline)
    with open(str(ctmfile), "w") as outf:
        for line in outlines:
            outf.write(" ".join(line) + "\n")

In [90]:
import json
from pathlib import Path
def json_ctm(filename, edit = False):
    with open(filename) as inf:
        data = json.load(inf)
        fileid = Path(filename).stem.replace(".w2v", "")
        for chunk in data['chunks']:
            start = chunk['timestamp'][0]
            end = chunk['timestamp'][1]
            word = chunk['text'].lower()
            if edit:
                edit = f" {word} cor"
            else:
                edit = ""
            print(f"{fileid} 1 {start} {end} {word} 1.0{edit}")

In [ ]:
json_ctm('/Users/joregan/Playing/hsi/audio/wav2vec/hsi_7_0719_222_003_main.w2v.json')

In [88]:
def json_ctm_path(fid, ctmedit=True):
    return json_ctm(f"/Users/joregan/Playing/hsi/audio/wav2vec/{fid}.w2v.json", ctmedit)

In [110]:
json_ctm_path("hsi_5_0718_209_001_inter", True)

hsi_5_0718_209_001_inter 1 0.86 1.24 start 1.0 start cor
hsi_5_0718_209_001_inter 1 3.14 3.7 recording 1.0 recording cor
hsi_5_0718_209_001_inter 1 3.8 4.02 now 1.0 now cor
hsi_5_0718_209_001_inter 1 6.84 7.08 olke 1.0 olke cor
hsi_5_0718_209_001_inter 1 7.58 7.7 no 1.0 no cor
hsi_5_0718_209_001_inter 1 10.56 10.64 ah 1.0 ah cor
hsi_5_0718_209_001_inter 1 10.76 10.84 no 1.0 no cor
hsi_5_0718_209_001_inter 1 10.96 10.98 i 1.0 i cor
hsi_5_0718_209_001_inter 1 11.06 11.2 think 1.0 think cor
hsi_5_0718_209_001_inter 1 11.28 11.34 it 1.0 it cor
hsi_5_0718_209_001_inter 1 11.38 11.52 would 1.0 would cor
hsi_5_0718_209_001_inter 1 11.54 11.6 be 1.0 be cor
hsi_5_0718_209_001_inter 1 11.64 11.88 better 1.0 better cor
hsi_5_0718_209_001_inter 1 11.96 12.08 not 1.0 not cor
hsi_5_0718_209_001_inter 1 12.14 12.2 to 1.0 to cor
hsi_5_0718_209_001_inter 1 12.24 12.42 wark 1.0 wark cor
hsi_5_0718_209_001_inter 1 12.48 12.62 that 1.0 that cor
hsi_5_0718_209_001_inter 1 12.72 12.92 much 1.0 much cor
hsi_

In [101]:
def whisperx_json_ctm(filepath, ctmedit=False):
    with open(filepath) as inf:
        data = json.load(inf)
    fileid = Path(filepath).stem
    for segment in data['segments']:
        for word in segment['words']:
            if "speaker" in word:
                speaker = word['speaker']
                channel = int(speaker.replace("SPEAKER_0", "")) + 1
                start = word['start']
                end = word['end']
            else:
                channel = 1
                start = 0.0
                end = 0.0
            text = word['word']
            conf = word['score']
            norm = normword(text)
            if ctmedit:
                edit = f" {text} cor"
            else:
                edit = ""
            print(f"{fileid} {channel} {start} {end} {norm} {conf}{edit}")

In [97]:
def my_whisperx_json_ctm(a, b=True):
    whisperx_json_ctm(f"/Users/joregan/Playing/hsi/audio/whisperx-json/{a}.json", b)

In [111]:
my_whisperx_json_ctm("hsi_5_0718_209_001_inter")

hsi_5_0718_209_001_inter 1 0.879 1.3 start 0.921 Start cor
hsi_5_0718_209_001_inter 1 3.141 3.742 recording 0.942 recording cor
hsi_5_0718_209_001_inter 1 3.782 4.102 now 0.928 now. cor
hsi_5_0718_209_001_inter 1 6.825 7.205 okay 0.703 Okay, cor
hsi_5_0718_209_001_inter 1 7.605 10.648 now 0.962 now. cor
hsi_5_0718_209_001_inter 1 10.768 10.889 no 1.0 No, cor
hsi_5_0718_209_001_inter 1 10.969 11.009 i 1.0 I cor
hsi_5_0718_209_001_inter 1 11.069 11.229 think 1.0 think cor
hsi_5_0718_209_001_inter 1 11.289 11.369 it 0.883 it cor
hsi_5_0718_209_001_inter 1 11.389 11.549 would 0.737 would cor
hsi_5_0718_209_001_inter 1 11.569 11.629 be 1.0 be cor
hsi_5_0718_209_001_inter 1 11.649 11.929 better 0.889 better cor
hsi_5_0718_209_001_inter 1 11.97 12.11 not 1.0 not cor
hsi_5_0718_209_001_inter 1 12.15 12.21 to 1.0 to cor
hsi_5_0718_209_001_inter 1 12.25 12.45 walk 0.99 walk cor
hsi_5_0718_209_001_inter 1 12.49 12.65 that 1.0 that cor
hsi_5_0718_209_001_inter 1 12.73 12.95 much 0.909 much. cor
hs

In [107]:
def ctmedit_to_tsv(fileid):
    filename = f"/Users/joregan/Playing/hsi_ctmedit/{fileid}.txt"
    outfile = f"/tmp/{fileid}.tsv"
    with open(filename) as inf, open(outfile, "w") as outf:
        lastend = 0.0
        for line in inf.readlines():
            line = line.strip()
            if line == "":
                outf.write("\n")
                continue
            parts = line.split()
            lastend = float(parts[3])
            if line.endswith(" cor"):
                outf.write(f"{parts[2]}\t{parts[3]}\t{parts[6]}\n")
            elif line.endswith(" sub"):
                outf.write(f"{parts[2]}\t{parts[3]}\t{parts[4]}/{parts[6]}\n")
            elif line.endswith(" ins"):
                outf.write(f"{parts[2]}\t{parts[3]}\t[{parts[4]}]\n")
            elif line.endswith(" del"):
                outf.write(f"{lastend}\t{lastend + 0.1}\t[{parts[4]}]\n")

In [113]:
ctmedit_to_tsv("hsi_5_0718_209_001_inter")

In [115]:
def set_to_left(text):
    lines = text.split("\n")
    for line in lines:
        if line == "":
            continue
        parts = line.split(" ")
        parts[6] = parts[4]
        parts[7] = "cor"
        print(" ".join(parts))

In [116]:
text = """
hsi_5_0718_209_001_inter 1 209.64 209.82 can 1.0 do sub
hsi_5_0718_209_001_inter 1 209.82 209.91 you 1.0 you cor
hsi_5_0718_209_001_inter 1 209.91 210.3 just 1.0 - ins
hsi_5_0718_209_001_inter 1 210.6 211.05 expand 1.0 - ins
hsi_5_0718_209_001_inter 1 211.05 211.2 with 0.664748 - ins
hsi_5_0718_209_001_inter 1 211.2 211.38 about 1.0 - ins
hsi_5_0718_209_001_inter 1 211.38 211.5 your 1.0 - ins
hsi_5_0718_209_001_inter 1 211.5 212.19 paintings 1.0 - ins
"""
set_to_left(text)

hsi_5_0718_209_001_inter 1 209.64 209.82 can 1.0 can cor
hsi_5_0718_209_001_inter 1 209.82 209.91 you 1.0 you cor
hsi_5_0718_209_001_inter 1 209.91 210.3 just 1.0 just cor
hsi_5_0718_209_001_inter 1 210.6 211.05 expand 1.0 expand cor
hsi_5_0718_209_001_inter 1 211.05 211.2 with 0.664748 with cor
hsi_5_0718_209_001_inter 1 211.2 211.38 about 1.0 about cor
hsi_5_0718_209_001_inter 1 211.38 211.5 your 1.0 your cor
hsi_5_0718_209_001_inter 1 211.5 212.19 paintings 1.0 paintings cor
